In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import umap
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
sns.set_palette("deep")

In [ ]:
df = pd.read_csv('player_data_for_clustering.csv')
print("Baseline Dataset Shape:", df.shape)
print("\nBaseline Dataset Head:")
print(df.head())

In [ ]:
features = ['Gls_per90', 'OffensiveImpact', 'Won_per90', 'Lost_per90', 'AerialEff',
            'AerialImpact', 'Tkl_per90', 'TklW_per90', 'DefImpact', 'Pass_Att_per90',
            'Pass_Cmp_per90', 'PassImpact', 'Takeons_Att_per90', 'Takeons_Succ_per90',
            'TakeonImpact']

scaler = StandardScaler()
X = scaler.fit_transform(df[features])

print("\nStandardized Feature Shape:", X.shape)
print("\nSample of Standardized Features:")
print(pd.DataFrame(X, columns=features).head())

In [ ]:
inertia = []
K = range(2, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 6))
plt.plot(K, inertia, 'bx-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal k')
plt.show()

print("\nInertia values for k=2 to 10:", inertia)

In [ ]:
k = 4
kmeans = KMeans(n_clusters=k, random_state=42)
cluster_labels = kmeans.fit_predict(X)

silhouette_avg = silhouette_score(X, cluster_labels)
print(f"\nSilhouette Score for k={k}: {silhouette_avg:.3f}")

df['Cluster'] = cluster_labels
print("\nSample of Players with Cluster Assignments:")
print(df[['Player_ID', 'Player', 'Cluster']].head(10))

In [ ]:
reducer = umap.UMAP(n_components=2, random_state=42)
embedding = reducer.fit_transform(X)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=cluster_labels, cmap='viridis')
plt.colorbar(scatter, label='Cluster')
plt.title('UMAP Projection of Player Clusters')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.show()

print("\nUMAP embedding shape:", embedding.shape)

In [ ]:
from scipy.spatial import distance_matrix

def find_similar_players(player_id, df, features, n_similar=5):
    player_data = df[df['Player_ID'] == player_id][features].values
    if len(player_data) == 0:
        return "Player not found"

    distances = distance_matrix(player_data, df[features].values)[0]
    similar_indices = distances.argsort()[1:n_similar + 1]
    similar_players = df.iloc[similar_indices][['Player_ID', 'Player', 'Team', 'Position', 'Cluster']]
    return similar_players.assign(Distance=distances[similar_indices])

player_id = '42fd9c7f'
similar_players = find_similar_players(player_id, df, features)
print(f"\nSimilar Players to {df[df['Player_ID'] == player_id]['Player'].values[0]} (Player_ID: {player_id}):")
print(similar_players)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Gls_per90', y='OffensiveImpact', hue='Cluster', style='Cluster', s=100)
plt.scatter(df[df['Player_ID'] == player_id]['Gls_per90'], df[df['Player_ID'] == player_id]['OffensiveImpact'],
            color='red', s=200, label='Mbappé')
plt.legend()
plt.title('Player Similarity in Goals Cluster Space')
plt.show()